In [ ]:
from my_functions import * #??? ????????? ? ????. ??? ????? ???????????. ??????? ????? ?????, ??????? ? csv, ??????? ? ????? ????? pandas.read_csv ??? read_excel
import pandas as pd
from numpy import sin, cos, arccos, pi, round

def rad2deg(radians):
    degrees = radians * 180 / pi
    return degrees

def deg2rad(degrees):
    radians = degrees * pi / 180
    return radians

def getDistanceBetweenPointsNew(latitude1, longitude1, latitude2, longitude2, unit = 'kilometers'):
    
    theta = longitude1 - longitude2
    
    distance = 60 * 1.1515 * rad2deg(
        arccos(
            (sin(deg2rad(latitude1)) * sin(deg2rad(latitude2))) + 
            (cos(deg2rad(latitude1)) * cos(deg2rad(latitude2)) * cos(deg2rad(theta)))
        )
    )
    
    if unit == 'miles':
        return round(distance, 2)
    if unit == 'kilometers':
        return round(distance * 1.609344, 2)

In [ ]:
scr="""
select distinct firms.city_id "CityID",firms.latitude ,firms.longitude,firms.id "FirmID"
from firm.firms firms
left join firm.fact_rubrics rubrics on firms.id=rubrics.firm_id 
left join firm.dim_rubrics on rubrics.rubric_id =dim_rubrics.id
left join firm.dim_cities_dwh cities on firms.city_id =cities."CityID" 
where 1=1
	--and firms.city_id =1995
	and firms.city_id_2gis =cities.id 
	and firms.topic='wildberries'
"""
dots=df_build('delta_bi',scr)

In [ ]:
branches=create_branches()

In [ ]:
temp=branches.merge(dots,on='CityID',how='inner')
temp_city=temp.groupby(['CityID'],as_index=False).FirmID.nunique()
temp_city=temp_city.rename(columns={'FirmID':'city_cnt'})

temp['distance']=getDistanceBetweenPointsNew(temp.latitude_x,temp.longitude_x,temp.latitude_y,temp.longitude_y)
temp=temp.query('distance<=1').groupby(['BranchCode','CityID'],as_index=False).distance.count()
temp=temp.rename(columns={'distance':'km1_cnt'})

temp=branches.merge(temp,on=['BranchCode','CityID'],how='left').merge(temp_city,on='CityID',how='left').query('StoreName=="??????? ?????"')
temp['cut']=temp.km1_cnt/temp.city_cnt
temp=temp.fillna(0)
temp

In [ ]:
temp=branches.merge(dots,on='CityID',how='inner')
temp.query('CityID==1995').FirmID.unique()

In [ ]:
features_orig=df_build('dns_dwh','lookalike_filials_feature')
features_orig.date_open=features_orig.date_open.astype('datetime64[ns]')

competitors=df_build('dns_dwh','lookalike_filials_competitors')

In [ ]:
branches_dns=features_orig.groupby('CityID').BranchCode.count().reset_index()
branches_dns=branches_dns.rename(columns={'BranchCode':'branches_dns_in_city'})
features_orig=features_orig.merge(branches_dns,how='left',on='CityID')
del branches_dns

In [ ]:
# ?????? ??????? ??? ??????????
distance=features_orig.merge(features_orig[['longitude','latitude']],how='cross')
distance['distance']=getDistanceBetweenPointsNew(distance.latitude_x,distance.longitude_x,distance.latitude_y,distance.longitude_y)

# ?????? ???? - ????? ????????? ??? ???????
temp=distance.query('distance>0 and distance<=1')
temp=temp.groupby('BranchCode').BranchID.count().reset_index()
temp=temp.rename(columns={'BranchID':'other_branches_dns_in_1km'})
features_orig=features_orig.merge(temp,how='left',on='BranchCode')
features_orig.other_branches_dns_in_1km=features_orig.other_branches_dns_in_1km.fillna(0)

temp=distance.query('distance>0 and distance<=3')
temp=temp.groupby('BranchCode').BranchID.count().reset_index()
temp=temp.rename(columns={'BranchID':'other_branches_dns_in_3km'})
features_orig=features_orig.merge(temp,how='left',on='BranchCode')
features_orig.other_branches_dns_in_3km=features_orig.other_branches_dns_in_3km.fillna(0)


In [ ]:
# ?????????? ?? ???
features_orig['distance_to_rrc']=getDistanceBetweenPointsNew(features_orig.store_latitude,features_orig.store_longitude,features_orig.latitude,features_orig.longitude)


In [ ]:
# ??????? ??????????? ??????????

distance=features_orig[['BranchCode','longitude','latitude']].merge(competitors[['longitude','latitude']],how='cross')
distance['distance']=getDistanceBetweenPointsNew(distance.latitude_x,distance.longitude_x,distance.latitude_y,distance.longitude_y)


temp=distance.query('distance<=1')
temp=temp.groupby('BranchCode').longitude_x.count().reset_index()
temp=temp.rename(columns={'longitude_x':'competitors_in_1km'})
features_orig=features_orig.merge(temp,how='left',on='BranchCode')
features_orig.competitors_in_1km=features_orig.competitors_in_1km.fillna(0)

temp=distance.query('distance<=3')
temp=temp.groupby('BranchCode').longitude_x.count().reset_index()
temp=temp.rename(columns={'longitude_x':'competitors_in_3km'})
features_orig=features_orig.merge(temp,how='left',on='BranchCode')
features_orig.competitors_in_3km=features_orig.competitors_in_3km.fillna(0)